In [2]:
pip install cirq

  Using cached cirq-1.6.1-py3-none-any.whl.metadata (16 kB)
  Using cached cirq_aqt-1.6.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached cirq_core-1.6.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached cirq_google-1.6.1-py3-none-any.whl.metadata (4.9 kB)
  Using cached cirq_ionq-1.6.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached cirq_pasqal-1.6.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached cirq_web-1.6.1-py3-none-any.whl.metadata (5.4 kB)
  Using cached google_api_core-2.30.3-py3-none-any.whl.metadata (3.1 kB)
  Using cached proto_plus-1.28.0-py3-none-any.whl.metadata (2.2 kB)
  Using cached protobuf-5.29.6-cp310-abi3-win_amd64.whl.metadata (592 bytes)
  Using cached typedunits-0.0.2-cp314-cp314-win_amd64.whl.metadata (5.1 kB)
  Using cached googleapis_common_protos-1.75.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached google_auth-2.51.0-py3-none-any.whl.metadata (5.5 kB)
  Using cached grpcio-1.80.0-cp314-cp314-win_amd64.whl.metadata (3.9 kB)
  Using cached grpcio_st


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
"""
Quantum Capsule Network — Cirq
S-Score Computation Only
=================================
Cirq equivalent of the Qiskit S-score script.
Computes QFIM-based S-scores across adaptive depth growth,
saves results to CSV, prints per-step timing.
"""

import numpy as np
import time
import warnings
import csv
import os
from datetime import datetime
warnings.filterwarnings('ignore')

import cirq
from scipy.linalg import eigvalsh

np.random.seed(42)

# ── Hyperparameters ───────────────────────────────────────────────────────────
N_CAPS        = 4
CAP_SIZE      = 3
INIT_DEPTH    = 2
MAX_DEPTH     = 8
PLATEAU_THR   = 0.30
QFIM_INTERVAL = 5
EPOCHS_MAX    = 100
LOSS_THR      = 0.18

timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_CSV = f"cirq_sscore_records_{timestamp}.csv"

print("=" * 60)
print("  QCN S-Score — Cirq")
print(f"  Caps={N_CAPS}x{CAP_SIZE}  MaxDepth={MAX_DEPTH}")
print(f"  Output → {OUTPUT_CSV}")
print("=" * 60)

# ── Cirq circuit for one capsule ──────────────────────────────────────────────

def build_capsule_cirq(cap_size, depth, params_np):
    """
    Build and simulate a single capsule circuit in Cirq.
    Returns the final statevector.
    params_np: flat array of length depth*cap_size*4
    """
    qubits = cirq.LineQubit.range(cap_size)
    ops    = []

    # Initial Hadamard
    for q in qubits:
        ops.append(cirq.H(q))

    idx = 0
    for layer in range(depth):
        # RY layer
        for i in range(cap_size):
            ops.append(cirq.ry(params_np[idx])(qubits[i]))
            idx += 1
        # RZ-RY-RZ per qubit
        for i in range(cap_size):
            ops.append(cirq.rz(params_np[idx])(qubits[i]));  idx += 1
            ops.append(cirq.ry(params_np[idx])(qubits[i]));  idx += 1
            ops.append(cirq.rz(params_np[idx])(qubits[i]));  idx += 1
        # CNOT entanglement
        if layer % 2 == 0:
            for i in range(cap_size - 1):
                ops.append(cirq.CNOT(qubits[i], qubits[i + 1]))
            if cap_size > 2:
                ops.append(cirq.CNOT(qubits[-1], qubits[0]))

    circuit = cirq.Circuit(ops)
    sim     = cirq.Simulator()
    result  = sim.simulate(circuit)
    return result.final_state_vector


def build_full_qcn_cirq(n_caps, cap_size, depths, all_params):
    """
    Build full QCN circuit in Cirq and return statevector.
    all_params: list of param arrays, one per capsule.
    """
    n      = n_caps * cap_size
    qubits = cirq.LineQubit.range(n)
    ops    = []

    for q in qubits:
        ops.append(cirq.H(q))

    max_d = max(depths)
    for layer in range(max_d):
        for c in range(n_caps):
            if layer >= depths[c]:
                continue
            w   = list(range(c * cap_size, (c + 1) * cap_size))
            prm = all_params[c]
            idx = layer * cap_size * 4

            for j, qi in enumerate(w):
                ops.append(cirq.ry(prm[idx + j])(qubits[qi]))
            for j, qi in enumerate(w):
                li = idx + cap_size + j * 3
                ops.append(cirq.rz(prm[li])(qubits[qi]))
                ops.append(cirq.ry(prm[li + 1])(qubits[qi]))
                ops.append(cirq.rz(prm[li + 2])(qubits[qi]))
            if layer % 2 == 0:
                for j in range(len(w) - 1):
                    ops.append(cirq.CNOT(qubits[w[j]], qubits[w[j + 1]]))
                if len(w) > 2:
                    ops.append(cirq.CNOT(qubits[w[-1]], qubits[w[0]]))

        if layer % 2 == 0:
            ic_offset = (layer // 2) % 2
            for c in range(ic_offset, n_caps - 1, 2):
                if layer < depths[c] and layer < depths[c + 1]:
                    ba = (c + 1) * cap_size - 1
                    bb = (c + 1) * cap_size
                    ops.append(cirq.CNOT(qubits[ba], qubits[bb]))

    circuit = cirq.Circuit(ops)
    sim     = cirq.Simulator()
    result  = sim.simulate(circuit)
    return result.final_state_vector

# ── S-score (QFIM) via Cirq ───────────────────────────────────────────────────

def compute_s_score_cirq(cap_params_np, depth, cap_size):
    n_p = len(cap_params_np)
    eps = 0.05

    def get_sv(p_vals):
        return build_capsule_cirq(cap_size, depth, p_vals)

    psi0 = get_sv(cap_params_np)
    dim  = len(psi0)
    J    = np.zeros((2 * dim, n_p))
    for i in range(n_p):
        p_p = cap_params_np.copy(); p_p[i] += eps
        p_m = cap_params_np.copy(); p_m[i] -= eps
        dpsi = (get_sv(p_p) - get_sv(p_m)) / (2 * eps)
        J[:dim, i] = np.real(dpsi)
        J[dim:, i] = np.imag(dpsi)

    qfim  = 4.0 * (J.T @ J)
    eigs  = np.clip(np.abs(eigvalsh(qfim)), 1e-8, 1e4)
    d_eff = float((np.sum(eigs) ** 2) / (np.sum(eigs ** 2) + 1e-12))
    kappa = float(np.max(eigs) / (np.min(eigs) + 1e-8))
    s     = d_eff / (np.log10(kappa + 1) + 1e-10)
    return float(s)

# ── Helpers ───────────────────────────────────────────────────────────────────

def ppc(depth, cap_size):
    return depth * cap_size * 4

def init_params(depth, cap_size, rng):
    p       = rng.uniform(0, 0.5, ppc(depth, cap_size))
    p[3::4] = rng.uniform(0.8, 1.2, len(p[3::4]))
    return p.astype(float)

# ── Adaptive run ──────────────────────────────────────────────────────────────

def run_adaptive_sscore_cirq():
    N_QUBITS = N_CAPS * CAP_SIZE
    rng    = np.random.RandomState(0)
    depths = [INIT_DEPTH] * N_CAPS
    cap_params = [init_params(INIT_DEPTH, CAP_SIZE, rng) for _ in range(N_CAPS)]
    s_window   = [[] for _ in range(N_CAPS)]
    growth_log = []
    records    = []

    rng_d   = np.random.RandomState(7)
    N_DATA  = 32
    X_train = rng_d.uniform(0, np.pi, (N_DATA, N_QUBITS))
    y_train = ((np.sum(X_train[:, :N_QUBITS//2], axis=1) >
                np.sum(X_train[:, N_QUBITS//2:], axis=1)) * 2 - 1).astype(float)

    def forward(all_params):
        sv    = build_full_qcn_cirq(N_CAPS, CAP_SIZE, depths, all_params)
        n     = N_CAPS * CAP_SIZE
        vn    = np.ones(N_CAPS) / N_CAPS
        outs  = []
        for c in range(N_CAPS):
            wire  = c * CAP_SIZE
            probs = np.abs(sv) ** 2
            z_exp = sum(probs[i] * (1 - 2 * ((i >> (n - 1 - wire)) & 1))
                        for i in range(len(sv)))
            outs.append(float(z_exp))
        return np.dot(vn, np.array(outs))

    def flat_params():
        return np.concatenate(cap_params)

    def unpack(flat):
        out, offset = [], 0
        for c in range(N_CAPS):
            sz = ppc(depths[c], CAP_SIZE)
            out.append(flat[offset: offset + sz].copy())
            offset += sz
        return out

    def set_flat(flat):
        up = unpack(flat)
        for c in range(N_CAPS):
            cap_params[c] = up[c]

    def loss(flat, Xb, yb):
        cp = unpack(flat)
        total = 0.0
        for xi, yi in zip(Xb, yb):
            total += (forward(cp) - float(yi)) ** 2
        return total / len(Xb)

    def grad_fd(flat, Xb, yb, eps=1e-3):
        g = np.zeros_like(flat)
        for i in range(len(flat)):
            pp = flat.copy(); pp[i] += eps
            pm = flat.copy(); pm[i] -= eps
            g[i] = (loss(pp, Xb, yb) - loss(pm, Xb, yb)) / (2 * eps)
        return g

    def adam_step(g, m, v, t, lr=0.025, b1=0.9, b2=0.999, eps_=1e-8):
        m  = b1 * m + (1 - b1) * g
        v  = b2 * v + (1 - b2) * g ** 2
        mh = m / (1 - b1 ** t)
        vh = v / (1 - b2 ** t)
        return lr * mh / (np.sqrt(vh) + eps_), m, v

    flat   = flat_params()
    m_adam = np.zeros_like(flat)
    v_adam = np.zeros_like(flat)
    t_adam = 0
    wall0  = time.time()

    print(f"\n  Running adaptive S-score loop (Cirq) ...")
    for epoch in range(EPOCHS_MAX):
        idx  = np.random.choice(N_DATA, min(6, N_DATA), replace=False)
        Xb, yb = X_train[idx], y_train[idx]

        flat   = flat_params()
        g      = grad_fd(flat, Xb, yb)
        t_adam += 1
        up, m_adam, v_adam = adam_step(g, m_adam, v_adam, t_adam)
        flat  -= up
        set_flat(flat)

        lv = loss(flat, Xb, yb)

        if epoch % QFIM_INTERVAL == 0:
            t_sscore_start = time.time()
            grew = False
            for c in range(N_CAPS):
                t_c = time.time()
                s   = compute_s_score_cirq(cap_params[c], depths[c], CAP_SIZE)
                elapsed_c = time.time() - t_c
                wall_s    = time.time() - wall0
                records.append({
                    'framework':     'cirq',
                    'epoch':         epoch,
                    'capsule':       c,
                    'depth':         depths[c],
                    's_score':       round(s, 6),
                    'loss':          round(lv, 6),
                    'sscore_time_s': round(elapsed_c, 4),
                    'wall_time_s':   round(wall_s, 4),
                })
                s_window[c].append(s)
                print(f"    Ep{epoch:3d} Cap{c} | depth={depths[c]} "
                      f"| S={s:.4f} | t={elapsed_c:.2f}s")

                if depths[c] < MAX_DEPTH and len(s_window[c]) >= 4:
                    dS = (s_window[c][-1] - s_window[c][-4]) / 4
                    if abs(dS) < PLATEAU_THR:
                        old_d = depths[c]
                        depths[c] += 1
                        rng2    = np.random.RandomState(epoch * 100 + c)
                        new_lyr = rng2.uniform(0, 0.1, CAP_SIZE * 4)
                        new_lyr[3::4] = rng2.uniform(0.9, 1.1, CAP_SIZE)
                        cap_params[c] = np.concatenate([cap_params[c], new_lyr])
                        growth_log.append((epoch, c, old_d, depths[c]))
                        grew = True
                        print(f"    [Grow] Ep{epoch:3d} Cap{c}: "
                              f"{old_d}->{depths[c]}  |dS|={abs(dS):.3f}")

            if grew:
                flat   = flat_params()
                m_adam = np.zeros_like(flat)
                v_adam = np.zeros_like(flat)
                t_adam = 0

            t_sscore = time.time() - t_sscore_start
            print(f"  → S-score batch at Ep{epoch}: {t_sscore:.2f}s total")

        if lv < LOSS_THR:
            print(f"  Converged at epoch {epoch+1} (loss={lv:.4f})")
            break

    total_wall = time.time() - wall0
    print(f"\n  Total wall time: {total_wall:.2f}s")
    return records, growth_log, total_wall

# ── Run ───────────────────────────────────────────────────────────────────────
t_global = time.time()
records, growth_log, total_wall = run_adaptive_sscore_cirq()

# ── Save CSV ──────────────────────────────────────────────────────────────────
fieldnames = ['framework','epoch','capsule','depth','s_score',
              'loss','sscore_time_s','wall_time_s']
with open(OUTPUT_CSV, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(records)

print(f"\n  CSV saved → {OUTPUT_CSV}  ({len(records)} rows)")
print(f"  Growth events: {growth_log}")
print(f"  Script total time: {time.time() - t_global:.2f}s")
print("Done")

  QCN S-Score — Cirq
  Caps=4x3  MaxDepth=8
  Output → cirq_sscore_records_20260507_163106.csv

  Running adaptive S-score loop (Cirq) ...
    Ep  0 Cap0 | depth=2 | S=0.9349 | t=0.52s
    Ep  0 Cap1 | depth=2 | S=0.9662 | t=0.57s
    Ep  0 Cap2 | depth=2 | S=0.9869 | t=0.51s
    Ep  0 Cap3 | depth=2 | S=1.0319 | t=0.48s
  → S-score batch at Ep0: 2.08s total
    Ep  5 Cap0 | depth=2 | S=0.9540 | t=0.47s
    Ep  5 Cap1 | depth=2 | S=0.9560 | t=0.47s
    Ep  5 Cap2 | depth=2 | S=1.0088 | t=0.47s
    Ep  5 Cap3 | depth=2 | S=1.0217 | t=0.47s
  → S-score batch at Ep5: 1.88s total
    Ep 10 Cap0 | depth=2 | S=0.9750 | t=0.47s
    Ep 10 Cap1 | depth=2 | S=0.9522 | t=0.48s
    Ep 10 Cap2 | depth=2 | S=1.0211 | t=0.48s
    Ep 10 Cap3 | depth=2 | S=1.0015 | t=0.63s
  → S-score batch at Ep10: 2.06s total
    Ep 15 Cap0 | depth=2 | S=0.9842 | t=0.47s
    [Grow] Ep 15 Cap0: 2->3  |dS|=0.012
    Ep 15 Cap1 | depth=2 | S=0.9469 | t=0.48s
    [Grow] Ep 15 Cap1: 2->3  |dS|=0.005
    Ep 15 Cap2 | depth

In [ ]:
pip install qiskit

In [ ]:
"""
Quantum Capsule Network — Qiskit
S-Score Computation Only
=================================
Computes QFIM-based S-scores across adaptive depth growth,
saves results to CSV, prints per-step timing.
"""

import numpy as np
import time
import warnings
import csv
import os
from datetime import datetime
warnings.filterwarnings('ignore')

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector
from scipy.linalg import eigvalsh

np.random.seed(42)

# ── Hyperparameters ───────────────────────────────────────────────────────────
N_CAPS        = 4
CAP_SIZE      = 3
INIT_DEPTH    = 2
MAX_DEPTH     = 8
PLATEAU_THR   = 0.30
QFIM_INTERVAL = 5
EPOCHS_MAX    = 100
LOSS_THR      = 0.18

timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_CSV = f"qiskit_sscore_records_{timestamp}.csv"

print("=" * 60)
print("  QCN S-Score — Qiskit")
print(f"  Caps={N_CAPS}x{CAP_SIZE}  MaxDepth={MAX_DEPTH}")
print(f"  Output → {OUTPUT_CSV}")
print("=" * 60)

# ── Circuit builders ──────────────────────────────────────────────────────────

def build_capsule_circuit(cap_size, depth, prefix='p'):
    n_params = depth * cap_size * 4
    params   = ParameterVector(prefix, n_params)
    qc       = QuantumCircuit(cap_size)
    for i in range(cap_size):
        qc.h(i)
    idx = 0
    for layer in range(depth):
        for i in range(cap_size):
            qc.ry(params[idx], i); idx += 1
        for i in range(cap_size):
            qc.rz(params[idx], i); idx += 1
            qc.ry(params[idx], i); idx += 1
            qc.rz(params[idx], i); idx += 1
        if layer % 2 == 0:
            for i in range(cap_size - 1):
                qc.cx(i, i + 1)
            if cap_size > 2:
                qc.cx(cap_size - 1, 0)
    return qc, params


def build_full_qcn_circuit(n_caps, cap_size, depths):
    n  = n_caps * cap_size
    qc = QuantumCircuit(n)
    for i in range(n):
        qc.h(i)
    cap_params = []
    for c in range(n_caps):
        n_p  = depths[c] * cap_size * 4
        pvec = ParameterVector(f'c{c}', n_p)
        cap_params.append(pvec)

    max_d = max(depths)
    for layer in range(max_d):
        for c in range(n_caps):
            if layer >= depths[c]:
                continue
            w    = list(range(c * cap_size, (c + 1) * cap_size))
            pvec = cap_params[c]
            idx  = layer * cap_size * 4
            for j, wire in enumerate(w):
                qc.ry(pvec[idx + j], wire)
            for j, wire in enumerate(w):
                li = idx + cap_size + j * 3
                qc.rz(pvec[li],     wire)
                qc.ry(pvec[li + 1], wire)
                qc.rz(pvec[li + 2], wire)
            if layer % 2 == 0:
                for j in range(len(w) - 1):
                    qc.cx(w[j], w[j + 1])
                if len(w) > 2:
                    qc.cx(w[-1], w[0])
        if layer % 2 == 0:
            ic_offset = (layer // 2) % 2
            for c in range(ic_offset, n_caps - 1, 2):
                if layer < depths[c] and layer < depths[c + 1]:
                    ba = (c + 1) * cap_size - 1
                    bb = (c + 1) * cap_size
                    qc.cx(ba, bb)
    return qc, cap_params

# ── S-score (QFIM) ────────────────────────────────────────────────────────────

def compute_s_score(cap_params_np, depth, cap_size, prefix='sq'):
    qc, pvec = build_capsule_circuit(cap_size, depth, prefix=prefix)
    n_p = len(cap_params_np)
    eps = 0.05

    def get_sv(p_vals):
        bound = qc.assign_parameters(dict(zip(pvec, p_vals)))
        return Statevector(bound).data

    psi0 = get_sv(cap_params_np)
    dim  = len(psi0)
    J    = np.zeros((2 * dim, n_p))
    for i in range(n_p):
        p_p = cap_params_np.copy(); p_p[i] += eps
        p_m = cap_params_np.copy(); p_m[i] -= eps
        dpsi = (get_sv(p_p) - get_sv(p_m)) / (2 * eps)
        J[:dim, i] = np.real(dpsi)
        J[dim:, i] = np.imag(dpsi)

    qfim  = 4.0 * (J.T @ J)
    eigs  = np.clip(np.abs(eigvalsh(qfim)), 1e-8, 1e4)
    d_eff = float((np.sum(eigs) ** 2) / (np.sum(eigs ** 2) + 1e-12))
    kappa = float(np.max(eigs) / (np.min(eigs) + 1e-8))
    s     = d_eff / (np.log10(kappa + 1) + 1e-10)
    return float(s)

# ── Helpers ───────────────────────────────────────────────────────────────────

def ppc(depth, cap_size):
    return depth * cap_size * 4

def init_params(depth, cap_size, rng):
    p       = rng.uniform(0, 0.5, ppc(depth, cap_size))
    p[3::4] = rng.uniform(0.8, 1.2, len(p[3::4]))
    return p.astype(float)

# ── Full adaptive QCN pass (forward + depth growth, S-score only) ─────────────

def run_adaptive_sscore():
    rng    = np.random.RandomState(0)
    depths = [INIT_DEPTH] * N_CAPS
    cap_params = [init_params(INIT_DEPTH, CAP_SIZE, rng) for _ in range(N_CAPS)]
    s_window   = [[] for _ in range(N_CAPS)]
    growth_log = []
    records    = []          # (epoch, capsule, depth, s_score, wall_s)

    # simple dataset
    rng_d   = np.random.RandomState(7)
    N_DATA  = 32
    N_QUBITS = N_CAPS * CAP_SIZE
    X_train = rng_d.uniform(0, np.pi, (N_DATA, N_QUBITS))
    y_train = ((np.sum(X_train[:, :N_QUBITS//2], axis=1) >
                np.sum(X_train[:, N_QUBITS//2:], axis=1)) * 2 - 1).astype(float)

    # forward pass (simplified loss)
    def forward(flat):
        qc, cp = build_full_qcn_circuit(N_CAPS, CAP_SIZE, depths)
        bind   = {}
        offset = 0
        for c in range(N_CAPS):
            pvec = cp[c]
            sz   = ppc(depths[c], CAP_SIZE)
            for pi, pv in enumerate(pvec):
                bind[pv] = float(flat[offset + pi])
            offset += sz
        sv    = Statevector(qc.assign_parameters(bind)).data
        vn    = np.ones(N_CAPS) / N_CAPS
        outs  = []
        for c in range(N_CAPS):
            wire  = c * CAP_SIZE
            probs = np.abs(sv) ** 2
            z_exp = sum(probs[i] * (1 - 2 * ((i >> (N_QUBITS - 1 - wire)) & 1))
                        for i in range(len(sv)))
            outs.append(float(z_exp))
        return np.dot(vn, np.array(outs))

    def flat_params():
        return np.concatenate(cap_params)

    def set_flat(flat):
        offset = 0
        for c in range(N_CAPS):
            sz = ppc(depths[c], CAP_SIZE)
            cap_params[c] = flat[offset: offset + sz].copy()
            offset += sz

    def loss(flat, Xb, yb):
        total = 0.0
        for xi, yi in zip(Xb, yb):
            total += (forward(flat) - float(yi)) ** 2
        return total / len(Xb)

    def grad_fd(flat, Xb, yb, eps=1e-3):
        g = np.zeros_like(flat)
        for i in range(len(flat)):
            pp = flat.copy(); pp[i] += eps
            pm = flat.copy(); pm[i] -= eps
            g[i] = (loss(pp, Xb, yb) - loss(pm, Xb, yb)) / (2 * eps)
        return g

    def adam_step(g, m, v, t, lr=0.025, b1=0.9, b2=0.999, eps=1e-8):
        m  = b1 * m + (1 - b1) * g
        v  = b2 * v + (1 - b2) * g ** 2
        mh = m / (1 - b1 ** t)
        vh = v / (1 - b2 ** t)
        return lr * mh / (np.sqrt(vh) + eps), m, v

    flat   = flat_params()
    m_adam = np.zeros_like(flat)
    v_adam = np.zeros_like(flat)
    t_adam = 0
    wall0  = time.time()

    print(f"\n  Running adaptive S-score loop ...")
    for epoch in range(EPOCHS_MAX):
        t_ep = time.time()

        idx  = np.random.choice(N_DATA, min(6, N_DATA), replace=False)
        Xb, yb = X_train[idx], y_train[idx]

        flat   = flat_params()
        g      = grad_fd(flat, Xb, yb)
        t_adam += 1
        up, m_adam, v_adam = adam_step(g, m_adam, v_adam, t_adam)
        flat  -= up
        set_flat(flat)

        lv = loss(flat, Xb, yb)

        # ── S-score every QFIM_INTERVAL epochs ──
        if epoch % QFIM_INTERVAL == 0:
            t_sscore_start = time.time()
            grew = False
            for c in range(N_CAPS):
                t_c = time.time()
                s   = compute_s_score(cap_params[c], depths[c], CAP_SIZE,
                                      prefix=f's{epoch}c{c}')
                elapsed_c = time.time() - t_c
                wall_s    = time.time() - wall0
                records.append({
                    'framework': 'qiskit',
                    'epoch':     epoch,
                    'capsule':   c,
                    'depth':     depths[c],
                    's_score':   round(s, 6),
                    'loss':      round(lv, 6),
                    'sscore_time_s': round(elapsed_c, 4),
                    'wall_time_s':   round(wall_s, 4),
                })
                s_window[c].append(s)
                print(f"    Ep{epoch:3d} Cap{c} | depth={depths[c]} "
                      f"| S={s:.4f} | t={elapsed_c:.2f}s")

                # growth check
                if depths[c] < MAX_DEPTH and len(s_window[c]) >= 4:
                    dS = (s_window[c][-1] - s_window[c][-4]) / 4
                    if abs(dS) < PLATEAU_THR:
                        old_d = depths[c]
                        depths[c] += 1
                        rng2     = np.random.RandomState(epoch * 100 + c)
                        new_lyr  = rng2.uniform(0, 0.1, CAP_SIZE * 4)
                        new_lyr[3::4] = rng2.uniform(0.9, 1.1, CAP_SIZE)
                        cap_params[c] = np.concatenate([cap_params[c], new_lyr])
                        growth_log.append((epoch, c, old_d, depths[c]))
                        grew = True
                        print(f"    [Grow] Ep{epoch:3d} Cap{c}: "
                              f"{old_d}->{depths[c]}  |dS|={abs(dS):.3f}")

            if grew:
                flat   = flat_params()
                m_adam = np.zeros_like(flat)
                v_adam = np.zeros_like(flat)
                t_adam = 0
            t_sscore = time.time() - t_sscore_start
            print(f"  → S-score batch at Ep{epoch}: {t_sscore:.2f}s total")

        if lv < LOSS_THR:
            print(f"  Converged at epoch {epoch+1} (loss={lv:.4f})")
            break

    total_wall = time.time() - wall0
    print(f"\n  Total wall time: {total_wall:.2f}s")
    return records, growth_log, total_wall

# ── Run ───────────────────────────────────────────────────────────────────────
t_global = time.time()
records, growth_log, total_wall = run_adaptive_sscore()

# ── Save CSV ──────────────────────────────────────────────────────────────────
fieldnames = ['framework','epoch','capsule','depth','s_score',
              'loss','sscore_time_s','wall_time_s']
with open(OUTPUT_CSV, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(records)

print(f"\n  CSV saved → {OUTPUT_CSV}  ({len(records)} rows)")
print(f"  Growth events: {growth_log}")
print(f"  Script total time: {time.time() - t_global:.2f}s")
print("Done")

  QCN S-Score — Qiskit
  Caps=4x3  MaxDepth=8
  Output → qiskit_sscore_records_20260507_195449.csv

  Running adaptive S-score loop ...
    Ep  0 Cap0 | depth=2 | S=0.9033 | t=0.50s
    Ep  0 Cap1 | depth=2 | S=0.9641 | t=0.89s
    Ep  0 Cap2 | depth=2 | S=0.9642 | t=1.00s
    Ep  0 Cap3 | depth=2 | S=1.0216 | t=0.65s
  → S-score batch at Ep0: 3.05s total
    Ep  5 Cap0 | depth=2 | S=0.8662 | t=0.25s
    Ep  5 Cap1 | depth=2 | S=0.9587 | t=0.26s
    Ep  5 Cap2 | depth=2 | S=0.9427 | t=0.27s
    Ep  5 Cap3 | depth=2 | S=1.0030 | t=0.26s
  → S-score batch at Ep5: 1.06s total
    Ep 10 Cap0 | depth=2 | S=0.8350 | t=0.24s
    Ep 10 Cap1 | depth=2 | S=0.9550 | t=0.27s
    Ep 10 Cap2 | depth=2 | S=0.9173 | t=0.25s
    Ep 10 Cap3 | depth=2 | S=0.9921 | t=0.26s
  → S-score batch at Ep10: 1.02s total
    Ep 15 Cap0 | depth=2 | S=0.8291 | t=0.24s
    [Grow] Ep 15 Cap0: 2->3  |dS|=0.019
    Ep 15 Cap1 | depth=2 | S=0.9551 | t=0.23s
    [Grow] Ep 15 Cap1: 2->3  |dS|=0.002
    Ep 15 Cap2 | depth=2 

In [ ]:
"""
QCN S-Score Plot — dual y-axis
================================
Reads CSV output from qcn_sscore_qiskit.py or qcn_sscore_cirq.py
and produces the S-score vs Epoch plot with Circuit Depth overlay.

Usage:
    python qcn_sscore_plot.py  --csv  <path_to_csv>  [--out  <output_png>]

If no --csv is given, the script auto-finds the most recent CSV
in the current directory (qiskit_sscore_*.csv or cirq_sscore_*.csv).
"""

import argparse
import glob
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines  as mlines

# ── CLI ───────────────────────────────────────────────────────────────────────
parser = argparse.ArgumentParser()
parser.add_argument('--csv', default=None,
                    help='Path to the s-score CSV file')
parser.add_argument('--out', default=None,
                    help='Output PNG path (auto-named if omitted)')
args = parser.parse_args()

# ── Find CSV ──────────────────────────────────────────────────────────────────
if args.csv:
    csv_path = args.csv
else:
    candidates = sorted(
        glob.glob('qiskit_sscore_*.csv') +
        glob.glob('cirq_sscore_*.csv')
    )
    if not candidates:
        sys.exit("No CSV found. Pass --csv <path>")
    csv_path = candidates[-1]   # most recent
    print(f"  Auto-selected: {csv_path}")

df = pd.read_csv(csv_path)
framework = df['framework'].iloc[0].capitalize()
print(f"  Framework : {framework}")
print(f"  Rows      : {len(df)}")
print(f"  Columns   : {list(df.columns)}")

# ── Pivot s_score and depth per capsule ──────────────────────────────────────
N_CAPS = df['capsule'].nunique()
caps   = sorted(df['capsule'].unique())

ss_pivot  = df.pivot_table(index='epoch', columns='capsule',
                           values='s_score', aggfunc='first')
dep_pivot = df.pivot_table(index='epoch', columns='capsule',
                           values='depth',   aggfunc='first')
ss_pivot.columns  = [f'cap{c}' for c in ss_pivot.columns]
dep_pivot.columns = [f'cap{c}' for c in dep_pivot.columns]

# ── Detect growth epochs ──────────────────────────────────────────────────────
growth_epochs = set()
prev_depths   = None
for epoch in sorted(dep_pivot.index):
    cur = tuple(dep_pivot.loc[epoch])
    if prev_depths is not None:
        if any(c > p for c, p in zip(cur, prev_depths) if not (np.isnan(c) or np.isnan(p))):
            growth_epochs.add(epoch)
    prev_depths = cur

# ── Colors ────────────────────────────────────────────────────────────────────
cap_colors = {
    'cap0': '#1f77b4',
    'cap1': '#d62728',
    'cap2': '#ff7f0e',
    'cap3': '#2ca02c',
}

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(12, 6))
ax2 = ax1.twinx()

# growth shading
for ge in growth_epochs:
    ax1.axvspan(ge - 0.5, ge + 0.5, color='#e07b00', alpha=0.10, zorder=0)
    ax1.axvline(ge, color='#e07b00', lw=0.9, alpha=0.45, ls=':', zorder=1)

# S-score lines (solid) on ax1
for c in caps:
    col    = f'cap{c}'
    color  = cap_colors[col]
    epochs = ss_pivot.index.values
    scores = ss_pivot[col].values
    ax1.plot(epochs, scores,
             color=color,
             marker='o', markersize=3.5,
             linewidth=1.6,
             label=f'Cap {c} S-score',
             zorder=4)

# Depth lines (dashed) on ax2
for c in caps:
    col   = f'cap{c}'
    color = cap_colors[col]
    ax2.step(dep_pivot.index, dep_pivot[col],
             where='post',
             color=color,
             linestyle='--',
             linewidth=1.5,
             alpha=0.85,
             label=f'Cap {c} depth',
             zorder=3)

# ── Axes ──────────────────────────────────────────────────────────────────────
max_epoch = df['epoch'].max()
ax1.set_xlim(0, max_epoch)
ax2.set_ylim(0, 11)

ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('QFIM S-Score', fontsize=12)
ax2.set_ylabel('Circuit Depth', fontsize=12)

ax1.grid(True, alpha=0.25)
ax1.set_facecolor('white')
fig.patch.set_facecolor('white')

# ── Title ─────────────────────────────────────────────────────────────────────
ax1.set_title(f'QCN S-Score & Circuit Depth vs Epoch  [{framework}]',
              fontsize=13, fontweight='bold', pad=10)

# ── Legend ────────────────────────────────────────────────────────────────────
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

growth_patch = mpatches.Patch(color='#e07b00', alpha=0.35,
                               label='Depth growth event')
ax1.legend(handles1 + handles2 + [growth_patch],
           labels1  + labels2  + ['Depth growth event'],
           loc='upper left', fontsize=8, ncol=2, framealpha=0.90)

# ── Timing annotation ─────────────────────────────────────────────────────────
if 'sscore_time_s' in df.columns:
    mean_t = df['sscore_time_s'].mean()
    total_t = df['sscore_time_s'].sum()
    ax1.annotate(
        f"Mean S-score time: {mean_t:.2f}s/cap\nTotal S-score time: {total_t:.1f}s",
        xy=(0.99, 0.03), xycoords='axes fraction',
        fontsize=7.5, ha='right', va='bottom',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7))

plt.tight_layout()

# ── Save ──────────────────────────────────────────────────────────────────────
if args.out:
    out_path = args.out
else:
    base = os.path.splitext(csv_path)[0]
    out_path = base + '_plot.png'

plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f"\n  Plot saved → {out_path}")